# ДЛП для открытых сот: баланс сил «дно ячейки ↔ кромки стенок»

Отдельный ноутбук к статье «Теорема Нётер и сотовый двигатель Казимира», раздел «Ограничения», п. (а).

**Вопрос.** У замкнутой ячейки давление вакуума на дно уравновешено крышкой куба. Чем уравновешено давление на дно у *открытой* ячейки высотой $h$ — и уравновешено ли вообще?

**Модель (простейшая, но с той же геометрией).**
* 2D скалярное поле, $\hbar=c=1$, условия Дирихле на всех проводниках.
* Плоскость $z=0$ — пластина. Сверху — гребёнка: стенки нулевой толщины при $x=ja$, $0<z<h$, период $a=1$. Над $z=h$ — свободное полупространство. Снизу пластины — свободное полупространство (гладкая сторона).
* Метод ДЛП: евклидова функция Грина на мнимых частотах $\omega=i\xi$, сила — интеграл $\Delta\langle T_{zz}\rangle=\langle T_{zz}\rangle-\langle T_{zz}\rangle_{\text{своб}}$.

**План.**
1. Матрица отражения моды волновода от открытого конца (сшивка мод с волнами Флоке) и её проверки.
2. Давление внутри ячейки $p_{\text{cell}}(h)$ = давление на дно; концевая поправка как функция $h$.
3. $\Delta\langle T_{zz}\rangle$ над гребёнкой и под гладкой стороной.
4. Баланс: сила на дно, сила на кромки, полная сила.

Ядро: SageMath 9.6 (код — чистый Python + numpy; препарсер Sage отключается в первой ячейке).

In [ ]:
try:
    preparser(False)          # SageMath: выполнять ячейки как обычный Python
except NameError:
    pass
import numpy as np, time
pi = np.pi
J1 = complex(0, 1)
def trap(y, x):
    y = np.asarray(y); x = np.asarray(x)
    return float(np.sum((y[1:] + y[:-1]) * np.diff(x)) / 2.0)
def check(name, residual, tol):
    r = float(abs(residual))
    print(('OK  ' if r < tol else 'FAIL') + '  %-62s residual = %.3e (tol %.0e)' % (name, r, tol))

## 1. Матрица отражения открытого конца

Внутри ячейки ($0<x<1$, $z<h$) моды волновода $\sin(\alpha_m x)$, $\alpha_m=m\pi$, $q_m=\sqrt{\alpha_m^2+\xi^2}$. Над гребёнкой — волны Флоке $e^{ik_nx}$, $k_n=k_x+2\pi n$, $\kappa_n=\sqrt{k_n^2+\xi^2}$, затухающие вверх.

Падающая на конец мода $e^{q_m(z-h)}\sin\alpha_m x$ отражается в $\sum_{m'}R_{m'm}e^{-q_{m'}(z-h)}\sin\alpha_{m'}x$. Сшивка $\varphi$ и $\partial_z\varphi$ на $z=h$ (на стенках $\varphi=0$, щель — вся ячейка) даёт
$$R=(\mathbb 1+W)^{-1}(W-\mathbb 1),\qquad W_{pm}=2\sum_n\frac{I_{pn}\,\bar I_{mn}}{\kappa_n}\,q_m,\qquad I_{pn}=\int_0^1\sin(\alpha_px)\,e^{ik_nx}dx .$$
Проверки: (а) взаимность — при $k_x=0$ матрица $Q^{1/2}RQ^{-1/2}$ симметрична; (б) эрмитовость той же матрицы при любом $k_x$ (на мнимых частотах нет потерь и нет распространяющихся волн); (в) $R\to0$ при $\xi\to\infty$.

In [ ]:
def Imat(alpha, beta):
    A = alpha[:, None]; B = beta[None, :]
    sgn = ((-1.0) ** np.arange(1, len(alpha) + 1))[:, None]
    den = A**2 - B**2
    small = np.abs(den) < 1e-9
    num = A * (1 - sgn * np.exp(J1 * B))
    return np.where(small, 0.5 * J1 * np.sign(B) * np.ones_like(den), num / np.where(small, 1.0, den))

def R_open(kx, xi, M=30, N=60):
    alpha = np.arange(1, M + 1) * pi; q = np.sqrt(alpha**2 + xi**2)
    kn = kx + 2 * pi * np.arange(-N, N + 1); kap = np.sqrt(kn**2 + xi**2)
    I = Imat(alpha, kn); Jm = Imat(alpha, -kn).T
    W = 2.0 * (I / kap[None, :]) @ (Jm * q[None, :])
    return np.linalg.solve(np.eye(M) + W, W - np.eye(M)), q

def R_lid(kx, xi, M=30, N=60):                 # крышка: идеальное зеркало Дирихле
    q = np.sqrt((np.arange(1, M + 1) * pi)**2 + xi**2)
    return -np.eye(M), q

for kx in (0.0, 0.7):
    R, q = R_open(kx, 2.0)
    S = np.sqrt(q)[:, None] * R / np.sqrt(q)[None, :]
    if kx == 0.0: check('взаимность, k_x=0: |S - S^T|', np.abs(S - S.T).max(), 1e-12)
    check('эрмитовость, k_x=%.1f: |S - S^H|' % kx, np.abs(S - S.conj().T).max(), 1e-12)
for xi in (1.0, 10.0, 100.0):
    R, q = R_open(0.3, xi); print('xi = %6.1f   ||R||_2 = %.4f' % (xi, np.linalg.norm(R, 2)))

## 2. Давление внутри ячейки

**Объёмная часть.** Длинный волновод ширины $a=1$: давление $\frac1\pi\sum_{m\ge1}\int_0^\infty\frac{k^2}{2\omega_m}dk$ минус то же для свободного пространства. С обрезанием $e^{-\beta\omega}$ разность $=\dfrac{\zeta(3)}{16\pi}-\tfrac12 f(0)$, где $\tfrac12f(0)=\frac{1}{4\pi\beta^2}$ — обрезаниезависимый член (Эйлер–Маклорен, $m=0$ исключено условием Дирихле). Он одинаков на дне и на кромках и в баланс не входит; конечная часть — $\zeta(3)/16\pi$.

**Концевая поправка.** Мода между дном ($r_1=-1$) и открытым концом ($R$):
$$p_{\text{end}}(h)=-\frac1\pi\int_0^\infty d\xi\int_{-\pi}^{\pi}\frac{dk_x}{2\pi}\;\mathrm{Tr}\Big[Q\,X\Big],\qquad X=\mathcal M(\mathbb1-\mathcal M)^{-1},\quad \mathcal M=-PRP,\ P=\mathrm{diag}(e^{-q_mh}).$$
В пределе крышки ($R=-\mathbb1$) это обычная поправка Казимира для отрезка длины $h$; это проверяется ниже по точной формуле.

**Почему это давление на дно.** Внутри ячейки на стенках Дирихле $\varphi=0\Rightarrow\partial_z\varphi=0$, т.е. $T_{zx}=0$; в евклидовой функции Грина члены $e^{\pm q(z+z')}$ дают $\Delta T_{zz}=0$ тождественно, а члены $e^{\pm q(z-z')}$ от $z$ не зависят. Поэтому усреднённое $\Delta\langle T_{zz}\rangle$ в ячейке одинаково от дна до $z\to h^-$.

In [ ]:
def xi_grid(n=120, s=4.0):
    u = (np.arange(n) + 0.5) / n
    return s * np.tan(pi / 2 * u), s * pi / 2 / np.cos(pi / 2 * u)**2 / n

def p_end(h, R_fun, nxi=120, nk=16, M=30, N=60):
    xs, ws = xi_grid(nxi); ks = (np.arange(nk) + 0.5) / nk * 2 * pi - pi; tot = 0.0
    for xi, w in zip(xs, ws):
        acc = 0.0
        for kx in ks:
            R, q = R_fun(kx, xi, M, N)
            P = np.diag(np.exp(-q * h)); Mm = -P @ R @ P
            acc += np.trace(np.diag(q) @ Mm @ np.linalg.inv(np.eye(M) - Mm)).real
        tot += w * acc / nk
    return -tot / pi

# проверка на крышке: точная поправка Казимира для отрезка длины h, сумма по модам
def p_lid_exact(h, M=30):
    s = 0.0
    for m in range(1, M + 1):
        xs = np.linspace(0, 200, 200001)[1:]; q = np.sqrt((m * pi)**2 + xs**2)
        s += -trap(q * np.exp(-2 * q * h) / (1 - np.exp(-2 * q * h)), xs) / pi
    return s
for h in (0.5, 1.0):
    a_, b_ = p_end(h, R_lid, nk=1), p_lid_exact(h)
    check('крышка, h=%.1f: формула p_end = точная (отн.)' % h, (a_ - b_) / b_, 2e-3)

# объёмная часть: конечный член zeta(3)/(16 pi)
def f_cut(alpha, beta, n=20001):
    if alpha == 0: return 1 / (2 * pi * beta**2)
    smax = np.arccosh(max(60 / (beta * alpha), 1.0) + 1)
    s = np.linspace(0, smax, n); k = alpha * np.sinh(s); w = alpha * np.cosh(s)
    return trap(k**2 / 2 * np.exp(-beta * w), s) / pi
for beta in (0.1, 0.05, 0.025):
    Mx = int(60 / (beta * pi)) + 5
    S = sum(f_cut(pi * m, beta) for m in range(1, Mx))
    ms = np.linspace(0, Mx, 8 * Mx + 1); I = trap([f_cut(pi * m, beta, 4001) for m in ms], ms)
    print('beta = %.3f   (S - I) + f(0)/2 = %.6f    zeta(3)/16pi = %.6f' % (beta, S - I + f_cut(0, beta) / 2, 1.2020569 / (16 * pi)))

### Концевая поправка для открытой ячейки: зависимость от $h$ и сходимость

In [ ]:
p_bulk = 1.2020569 / (16 * pi)
hs = (0.25, 0.5, 1.0, 2.0)
t0 = time.time()
rows = {}
for (M, N, nxi, nk) in [(20, 40, 60, 8), (30, 60, 120, 16), (40, 80, 200, 24)]:
    rows[M] = [p_end(h, R_open, nxi=nxi, nk=nk, M=M, N=N) for h in hs]
    print('M=%2d N=%2d nxi=%3d nk=%2d :' % (M, N, nxi, nk), '  '.join('%.6f' % v for v in rows[M]))
print('время %.1f с' % (time.time() - t0))
print('\n   h      p_end(h)     p_cell = zeta(3)/16pi + p_end   (конечная часть давления на дно)')
for h, v in zip(hs, rows[40]):
    print('%5.2f   %10.6f   %10.6f' % (h, v, p_bulk + v))

## 3. Над гребёнкой и под гладкой стороной

**Над гребёнкой** ($z>h$) поле — сумма волн Флоке. Отражённая часть евклидовой функции Грина имеет вид $\sum_{n,n'}c_{nn'}e^{-\kappa_nz-\kappa_{n'}z'}e^{ik_nx-ik_{n'}x'}$. После усреднения по периоду остаются $n=n'$, и для каждого такого члена
$$\Delta T_{zz}\propto\partial_z\partial_{z'}-\partial_x\partial_{x'}-\partial_\tau\partial_{\tau'}\;\to\;\kappa_n^2-k_n^2-\xi^2=0 .$$
Следовательно, $\Delta\langle T_{zz}\rangle_{\text{над}}\equiv0$ **при любом** $z>h$, в том числе сразу над кромками.

**Под гладкой стороной** — то же с одной волной, $\Delta\langle T_{zz}\rangle_{\text{под}}\equiv0$.

Проверка тождества ниже — численно по случайным $k_n,\xi$ (это алгебра, но пусть будет).

In [ ]:
rng = np.random.default_rng(0)
kn = rng.uniform(-30, 30, 1000); xi = rng.uniform(0, 30, 1000); kap = np.sqrt(kn**2 + xi**2)
check('над гребёнкой: kappa^2 - k^2 - xi^2 = 0 для каждого порядка', np.max(np.abs(kap**2 - kn**2 - xi**2)), 1e-9)

## 4. Баланс сил на тело (на один период, вдоль $z$)

Сила на поверхность металла равна скачку $\Delta\langle T_{zz}\rangle$ на ней.

| где | $\Delta\langle T_{zz}\rangle$ сверху | снизу | сила |
|---|---|---|---|
| дно ячеек ($z=0^+$ / гладкая сторона $z=0^-$) | $p_{\text{cell}}(h)$ | $0$ | $+p_{\text{cell}}(h)$ на дно |
| кромки стенок ($z=h$) | $0$ (над гребёнкой) | $p_{\text{cell}}(h)$ (в ячейке) | $-p_{\text{cell}}(h)$ на кромки |
| боковые стенки | — | — | $0$ ($T_{zx}=0$) |

**Полная сила:** $p_{\text{cell}}(h)-p_{\text{cell}}(h)=0$ — при **любой** высоте $h$, при любом обрезании (обрезаниезависимая часть $p_{\text{cell}}$ тоже сокращается). Это и есть ответ на вопрос «чем уравновешено давление на дно»: кромками стенок. У замкнутой ячейки ту же роль играет крышка куба.

In [ ]:
print('   h     сила на дно (+z)   сила на кромки   полная сила')
for h, v in zip(hs, rows[40]):
    pc = p_bulk + v
    print('%5.2f      %+.6f        %+.6f       %+.1e' % (h, pc, -pc, pc - pc))

## Итог

* Давление вакуума на дно открытой ячейки **существует и зависит от $h$**: конечная часть $p_{\text{cell}}(h)=\zeta(3)/16\pi+p_{\text{end}}(h)$, причём $p_{\text{end}}$ убывает примерно как $e^{-2\pi h}$ (у ячеек высотой $h\gtrsim a$ концевая поправка уже мала).
* Ровно такая же сила с обратным знаком приложена к **кромкам стенок** — потому что над гребёнкой $\Delta\langle T_{zz}\rangle\equiv0$, а в ячейке $\Delta\langle T_{zz}\rangle$ от $z$ не зависит.
* **Полная сила на жёсткую пластину с открытыми сотами равна нулю** в равновесии при $T=0$ — в согласии с трансляционной инвариантностью (теорема Нётер для тела целиком).

**Ограничения модели:** 2D, скаляр, Дирихле, стенки нулевой толщины, $T=0$. Для 3D электромагнитного поля и идеального проводника аргумент тот же: на стенках $T_{zx}=0$, а для каждой отражённой волны Флоке над решёткой ТЕ и ТМ дают $\Delta T_{zz}=0$. Реальная $\varepsilon(\omega)$ меняет величину $p_{\text{cell}}$, но не баланс. Ненулевая сила требует выйти за рамки равновесия: разные температуры тела и окружения, невзаимные материалы, движение.